# Phase 1: Offline Distillation Data Generation (Hugging Face API)
Notebook này sinh dữ liệu ma trận khoảng cách $D$ giữa các rollouts giải toán để phục vụ quá trình Distillation cho Student Model.
**Được tối ưu hóa:** Gọi thẳng tới Hugging Face Router cho Llama-3.1-8B-Instruct. KHÔNG cần GPU, chạy trực tiếp trên CPU cá nhân hoặc Colab Basic.

In [ ]:
# [QUAN TRỌNG] Kết nối với Google Drive để đảm bảo lưu dữ liệu an toàn
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/Data_Phase1', exist_ok=True)
    print("Đã kết nối Google Drive thành công!")
except:
    print("Không chạy trên môi trường Colab.")

In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q datasets jsonlines tqdm requests

## Cấu hình API Key (Tuỳ chọn)
Mặc định hệ thống đã dùng HF_TOKEN được cung cấp sẵn. Nếu bạn muốn đổi sang token khác, hãy điền vào đây.

In [ ]:
import os
import phase1_distillation.config as config

# ĐIỀN API KEY CỦA BẠN VÀO ĐÂY (NẾU MUỐN ĐỔI)
# os.environ['HF_TOKEN'] = 'your_hf_token_here'
# config.HEADERS['Authorization'] = f"Bearer {os.environ['HF_TOKEN']}"

## Chạy Pipeline (Sinh K=4 Rollouts và Ma trận $D$)

In [ ]:
import jsonlines
from tqdm import tqdm
from phase1_distillation import MathDataset, MathRolloutGenerator, AlignmentJudge
from phase1_distillation.dataset import get_problem_id
import phase1_distillation.config as config
import os

os.makedirs(os.path.dirname(config.DRIVE_OUTPUT_FILE), exist_ok=True)

# 1. Khởi tạo Dataset (Bao gồm Resume logic)
dataset, processed_ids = MathDataset.load_with_resume(config.DRIVE_PROCESSED_IDS, sample_size=100)

# 2. Khởi tạo Generator & Judge (Sử dụng API)
generator = MathRolloutGenerator(model_id=config.MODEL_ID)
judge = AlignmentJudge(model_id=config.MODEL_ID)

# 3. Vòng lặp chính
with jsonlines.open(config.DRIVE_OUTPUT_FILE, mode='a') as writer, open(config.DRIVE_PROCESSED_IDS, "a") as track_file:
    # Truyền total vào tqdm để có thanh progress bar chính xác
    for item in tqdm(dataset, total=len(dataset), desc="Generating Distillation Data"):
        problem_id = get_problem_id(item['problem'])
        
        # Sinh 4 rollouts
        rollouts = generator.generate(item['problem'])
        
        # Sinh ma trận khoảng cách
        distance_matrices = judge.evaluate_pairs(item['problem'], rollouts)
        
        # Partial Saving Logic
        if distance_matrices is None or all(v is None for v in distance_matrices.values()):
            print(f"\n[!] Bỏ qua problem {problem_id} (Toàn bộ cặp đều fail). Sẽ retry lần sau.")
            continue
            
        # Ghi nối tiếp
        result = {
            "question_id": problem_id,
            "question": item['problem'],
            "ground_truth": item['ground_truth_solution'],
            "generated_rollouts": rollouts,
            "distance_matrices": distance_matrices
        }
        writer.write(result)
        
        # Cập nhật tracking
        track_file.write(problem_id + "\n")
        track_file.flush()
